<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

## 부록 D: 훈련 루프에 유용한 요소 추가하기

- 이 부록에서는 일반적인 사전훈련과 미세조정에서 사용되는 몇 가지 고급 기능을 훈련 함수에 추가합니다. 미세조정은 6장과 7장에서 다룹니다.
- 아래 세 섹션에서는 학습률 워밍업(learning rate warmup), 코사인 감쇠(cosine decay), 그리고 그래디언트 클리핑(gradient clipping)에 대해 논의합니다.
- 마지막 섹션에서는 이러한 기법들을 훈련 함수에 추가합니다.

- 5장의 코드를 재사용하여 모델을 초기화하는 것부터 시작합니다:

In [ ]:
from importlib.metadata import version
import torch

print("torch version:", version("torch"))


from previous_chapters import GPTModel
# 만약 `previous_chapters.py` 파일이 로컬에 없다면,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 예시:
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 단축된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 편향
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 참고:
# 다음 줄들의 주석을 해제하면 해당되는 경우 Apple Silicon 칩에서 코드가 실행되며,
# 이는 Apple CPU보다 약 2배 빠릅니다 (M3 MacBook Air에서 측정).
# 하지만 결과 손실 값이 약간 다를 수 있습니다.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # 추론 중 드롭아웃 비활성화

- 다음으로, 5장에서 사용한 동일한 코드를 사용하여 데이터 로더를 초기화합니다:

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
from previous_chapters import create_dataloader_v1
# 대안:
# from llms_from_scratch.ch02 import create_dataloader_v1


# 훈련/검증 비율
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

## D.1 학습률 워밍업

- LLM과 같은 복잡한 모델을 훈련할 때, 학습률 워밍업을 구현하면 훈련을 안정화하는 데 도움이 될 수 있습니다.
- 학습률 워밍업에서는 학습률을 매우 낮은 값(`initial_lr`)에서 사용자가 지정한 최대값(`peak_lr`)까지 점진적으로 증가시킵니다.
- 이렇게 하면 모델이 작은 가중치 업데이트로 훈련을 시작하게 되어, 훈련 중 큰 불안정화 업데이트의 위험을 줄이는 데 도움이 됩니다.

In [ ]:
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

- 일반적으로, 워밍업 단계의 수는 전체 단계 수의 0.1%에서 20% 사이입니다.
- 증분을 `peak_lr`과 `initial_lr`의 차이를 워밍업 단계 수로 나눈 값으로 계산할 수 있습니다.

In [ ]:
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps) # 20% 워밍업
print(warmup_steps)

- 인쇄된 책에서는 실수로 남겨진 코드 줄 `warmup_steps = 20`이 포함되어 있는데, 이는 사용되지 않으며 안전하게 무시할 수 있습니다.

In [ ]:
lr_increment = (peak_lr - initial_lr) / warmup_steps

global_step = -1
track_lrs = []

optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1
    
        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr
        
        # 계산된 학습률을 옵티마이저에 적용
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])
    
        # 손실 계산 및 가중치 업데이트
        # ...

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
total_training_steps = len(train_loader) * n_epochs
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("1.pdf")
plt.show()

## D.2 코사인 감쇠

- 복잡한 심층 신경망을 훈련하는 또 다른 인기 있는 기법은 코사인 감쇠로, 훈련 에포크에 걸쳐 학습률을 조정합니다.
- 코사인 감쇠에서 학습률은 코사인 곡선을 따라 초기값에서 거의 0에 가까운 값까지 반코사인 주기를 따라 감소합니다.
- 이 점진적 감소는 모델이 가중치를 개선하기 시작할 때 학습 속도를 늦추도록 설계되었습니다. 훈련이 진행됨에 따라 최솟값을 넘어서는 위험을 줄여주며, 이는 훈련 후반부의 안정화에 중요합니다.
- 코사인 감쇠는 학습률 조정에서 더 부드러운 전환을 위해 선형 감쇠보다 선호되는 경우가 많지만, 실제로는 선형 감쇠도 사용됩니다 (예: [OLMo: Accelerating the Science of Language Models](https://arxiv.org/abs/2402.00838)).

In [ ]:
import math

min_lr = 0.1 * initial_lr
track_lrs = []

lr_increment = (peak_lr - initial_lr) / warmup_steps
global_step = -1

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1
    
        # 현재 단계에 따라 학습률 조정 (워밍업 또는 코사인 어닐링)
        if global_step < warmup_steps:
            # 선형 워밍업
            lr = initial_lr + global_step * lr_increment  
        else:
            # 워밍업 후 코사인 어닐링
            progress = ((global_step - warmup_steps) / 
                        (total_training_steps - warmup_steps))
            lr = min_lr + (peak_lr - min_lr) * 0.5 * (
                1 + math.cos(math.pi * progress))
        
        # 계산된 학습률을 옵티마이저에 적용
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])
    
        # 손실 계산 및 가중치 업데이트

In [ ]:
plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("2.pdf")
plt.show()

## D.3 그래디언트 클리핑

- 그래디언트 클리핑은 LLM을 훈련할 때 훈련을 안정화하는 데 사용되는 또 다른 기법입니다.
- 임계값을 설정하여, 이 한계를 초과하는 그래디언트는 최대 크기로 축소되어 역전파 중 모델 매개변수에 대한 업데이트가 관리 가능한 범위 내에 유지되도록 합니다.
- 예를 들어, PyTorch의 `clip_grad_norm_` 메서드에서 `max_norm=1.0` 설정을 사용하면 그래디언트의 norm이 클리핑되어 최대 norm이 1.0을 초과하지 않습니다.
- "norm"은 모델의 매개변수 공간에서 그래디언트 벡터의 길이(또는 크기)의 측정을 의미합니다.
- 구체적으로, L2 norm이며, 유클리드 norm이라고도 합니다.
- 수학적으로, 성분 $\mathbf{v} = [v_1, v_2, \ldots, v_n]$을 가진 벡터 $\mathbf{v}$에 대해, L2 norm은 다음과 같이 정의됩니다:
$$
\| \mathbf{v} \|_2 = \sqrt{v_1^2 + v_2^2 + \ldots + v_n^2}
$$

- L2 norm은 행렬에 대해서도 유사하게 계산됩니다.
- 그래디언트 행렬이 다음과 같다고 가정해봅시다:
$$
G = \begin{bmatrix}
1 & 2 \\
2 & 4
\end{bmatrix}
$$

- 그리고 이 그래디언트를 `max_norm` 1로 클리핑하려고 합니다.

- 먼저, 이 그래디언트의 L2 norm을 계산합니다:
$$
\|G\|_2 = \sqrt{1^2 + 2^2 + 2^2 + 4^2} = \sqrt{25} = 5
$$

- $\|G\|_2 = 5$가 우리의 `max_norm` 1보다 크므로, 그래디언트의 norm이 정확히 1이 되도록 그래디언트를 축소해야 합니다. 스케일링 인수는 $\frac{max\_norm}{\|G\|_2} = \frac{1}{5}$로 계산됩니다.

- 따라서, 스케일된 그래디언트 행렬 $G'$는 다음과 같습니다:
$$
G' = \frac{1}{5} \times G = \begin{bmatrix}
\frac{1}{5} & \frac{2}{5} \\
\frac{2}{5} & \frac{4}{5}
\end{bmatrix}
$$

- 이를 실제로 살펴보겠습니다.
- 먼저, 일반적인 훈련 루프에서 하는 것처럼 새 모델을 초기화하고 훈련 배치에 대한 손실을 계산합니다.

In [ ]:
from previous_chapters import calc_loss_batch
# 대안:
# from llms_from_scratch.ch05 import calc_loss_batch


torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()

- `.backward()`를 호출하면, PyTorch는 그래디언트를 계산하여 각 가중치(매개변수) 행렬의 `.grad` 속성에 저장합니다.
- 모든 모델 가중치를 기준으로 가장 높은 그래디언트를 계산하는 유틸리티 함수를 정의해보겠습니다.

In [ ]:
def find_highest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = max_grad_param
    return max_grad

print(find_highest_gradient(model))

- 그래디언트 클리핑을 적용하면, 가장 큰 그래디언트가 이제 상당히 작아진 것을 볼 수 있습니다:

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(find_highest_gradient(model))

## D.4 수정된 훈련 함수

- 이제 위에서 다룬 세 가지 개념(학습률 워밍업, 코사인 감쇠, 그래디언트 클리핑)을 5장에서 다룬 `train_model_simple` 함수에 추가하여 아래의 더 정교한 `train_model` 함수를 만들어보겠습니다:

In [ ]:
from previous_chapters import evaluate_model, generate_and_print_sample
# 대안:
# from llms_from_scratch.ch05 import evaluate_model, generate_and_print_samplee


ORIG_BOOK_VERSION = False


def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1

    # 옵티마이저에서 최대 학습률 가져오기
    peak_lr = optimizer.param_groups[0]["lr"]

    # 훈련 과정의 총 반복 횟수 계산
    total_training_steps = len(train_loader) * n_epochs

    # 워밍업 단계 중 학습률 증분 계산
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            # 현재 단계에 따라 학습률 조정 (워밍업 또는 코사인 어닐링)
            if global_step < warmup_steps:
                # 선형 워밍업
                lr = initial_lr + global_step * lr_increment  
            else:
                # 워밍업 후 코사인 어닐링
                progress = ((global_step - warmup_steps) / 
                            (total_training_steps - warmup_steps))
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

            # 계산된 학습률을 옵티마이저에 적용
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)  # 현재 학습률 저장

            # 손실 계산 및 역전파
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            # 폭발하는 그래디언트를 피하기 위해 워밍업 단계 후 그래디언트 클리핑 적용
            if ORIG_BOOK_VERSION:
                if global_step > warmup_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
            else:
                if global_step >= warmup_steps:  # 책에서는 원래 global_step > warmup_steps를 사용했는데, 이로 인해 워밍업 후 클리핑 단계가 건너뛰어짐
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
            optimizer.step()
            tokens_seen += input_batch.numel()

            # 주기적으로 훈련 및 검증 세트에서 모델 평가
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader,
                    device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                # 현재 손실 출력
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        # 진행 상황을 모니터링하기 위해 모델에서 샘플 생성 및 출력
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
import tiktoken

# 참고:
# 실행 시간을 계산하려면 다음 코드의 주석을 해제하세요
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

peak_lr = 0.001  # 이것은 원래 책에서 실수로 5e-4로 설정되었음
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)  # 책에서 실수로 lr 할당이 누락됨
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 15
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device, n_epochs=n_epochs,
    eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=tokenizer, warmup_steps=warmup_steps, 
    initial_lr=1e-5, min_lr=1e-5
)

# 참고:
# 실행 시간을 표시하려면 다음 코드의 주석을 해제하세요
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

- 위의 결과를 보면, 모델이 처음에는 이해할 수 없는 단어 문자열을 생성하다가, 마지막에는 문법적으로 어느 정도 올바른 문장을 생성할 수 있게 됨을 알 수 있습니다.
- 마지막에 작성하는 몇 개의 구절을 확인해보면, 훈련 세트에 그대로 포함된 내용임을 알 수 있습니다 -- 단순히 훈련 데이터를 암기하는 것입니다.
- 여기서 과적합이 발생하는 이유는 매우 작은 훈련 세트를 가지고 있고, 그것을 너무 많이 반복하기 때문입니다.
  - 여기서의 LLM 훈련은 주로 교육 목적으로 사용됩니다. 주로 모델이 일관된 텍스트를 생성하는 법을 배울 수 있다는 것을 보고 싶습니다.
  - 값비싼 하드웨어에서 이 모델을 몇 주 또는 몇 달 동안 훈련하는 대신, 사전훈련된 가중치를 로드합니다.

- 학습률이 의도한 대로 동작하는지 빠르게 확인해보겠습니다.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(range(len(lrs)), lrs)
plt.ylabel("Learning rate")
plt.xlabel("Steps")
plt.show()

- 그리고 손실 곡선을 빠르게 살펴보겠습니다.

In [ ]:
from previous_chapters import plot_losses
# 대안:
# from llms_from_scratch.ch05 import plot_losses


epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout(); plt.savefig("3.pdf")
plt.show()

- 여기서 모델이 과적합되고 있음을 주목하세요. 이는 교육 목적으로 데이터셋을 매우 작게 유지했기 때문입니다 (랩톱 컴퓨터에서 코드를 실행할 수 있도록).
- 훨씬 더 큰 데이터셋에서의 더 긴 사전훈련 실행에 대해서는 [../../ch05/03_bonus_pretraining_on_gutenberg](../../ch05/03_bonus_pretraining_on_gutenberg)를 참조하세요.